1. Instalasi & Import Library

In [4]:
import pandas as pd
import numpy as np
import glob
import os


2. Load & Merge Dataset Mentah

In [6]:
# Cell 2: Load & Merge Dataset Mentah
# Menggunakan '../' karena notebook ini berada di folder 'notebooks'
path_raw = '../data/raw_x/'
all_files = glob.glob(os.path.join(path_raw, "*.csv"))

df_list = []
for file in all_files:
    try:
        df_temp = pd.read_csv(file)
        
        # Ekstrak nama file (tanpa .csv) untuk tahu dari file mana data ini berasal
        kategori_asal = os.path.basename(file).replace('.csv', '')
        df_temp['kategori_sumber'] = kategori_asal
        
        df_list.append(df_temp)
    except Exception as e:
        print(f"Gagal membaca file {file}: {e}")

# Gabungkan semua DataFrame menjadi satu
df_raw = pd.concat(df_list, ignore_index=True)

print(f"Total dataset mentah yang berhasil di-load: {df_raw.shape[0]} baris.")
display(df_raw.head(3))

Total dataset mentah yang berhasil di-load: 1873 baris.


,web_scraper_order,web_scraper_start_url,data,data2,data3,data4,data5,data6,data8,kategori_sumber,data7,data9,data10,data12,data11,data13
0,1775290311-1,https://x.com/search?q=%22PDAM%20mati%22%20OR%...,8m,Koh Alung,@alung_koh21471,krisis air,Pemerintah berupaya menangani,bersih di Karang Baru secara bertahap.,3,Air & Sanitasi,NaN,NaN,NaN,NaN,NaN,NaN
1,1775290311-2,https://x.com/search?q=%22PDAM%20mati%22%20OR%...,16m,Batampos,@BatamPos,"hingga Hari ke -70, Warga Tanjung Sengkuang Ma...",Krisis Air,NaN,5,Air & Sanitasi,NaN,NaN,NaN,NaN,NaN,NaN
2,1775290311-3,https://x.com/search?q=%22PDAM%20mati%22%20OR%...,17m,Rizki Fadillnaldo,@ciroalves2055,KRISIS AIR,NaN,BERSIH? JANGAN ASAL MENYALAHKAN!\nMasalahnya l...,3,Air & Sanitasi,NaN,NaN,NaN,NaN,NaN,NaN


3. Menyatukan Teks yang Terpecah

In [7]:
# Cell 3: Menyatukan Teks yang Terpecah
print("Memulai proses pembersihan dan penyatuan teks...")

# Identifikasi kolom waktu (biasanya bernama 'data')
time_col = 'data' if 'data' in df_raw.columns else None

# Ambil semua kolom teks hasil scraper (berawalan 'data' tapi bukan kolom waktu)
text_cols = [c for c in df_raw.columns if c.startswith('data') and c != time_col]

def bersihkan_dan_gabung(row):
    teks_gabungan = []
    username = 'Tidak diketahui'
    
    for col in text_cols:
        val = str(row[col]) if pd.notna(row[col]) else ''
        val = val.strip()
        if not val:
            continue
            
        # Jika kata berawalan @ dan berdiri sendiri, jadikan username
        if val.startswith('@') and len(val.split()) == 1:
            username = val
        # Jika bukan spam angka (seperti jumlah like/retweet), masukkan ke teks
        elif not (val.isdigit() and len(val) < 4):
            teks_gabungan.append(val)
            
    return pd.Series([username, ' '.join(teks_gabungan)])

# Terapkan fungsi ke seluruh baris
df_raw[['username', 'teks_bersih']] = df_raw.apply(bersihkan_dan_gabung, axis=1)

# Buat dataframe baru untuk data interim
df_clean = df_raw.copy()

# Pindahkan kolom waktu
if time_col:
    df_clean['waktu'] = df_clean[time_col]
else:
    df_clean['waktu'] = 'Tidak diketahui'

# Buang baris yang teksnya kosong setelah dibersihkan
df_clean = df_clean[df_clean['teks_bersih'] != '']

print(f"Total baris setelah teks disatukan & dibersihkan: {df_clean.shape[0]}")
display(df_clean[['waktu', 'username', 'teks_bersih']].head(3))

Memulai proses pembersihan dan penyatuan teks...
Total baris setelah teks disatukan & dibersihkan: 1863


,waktu,username,teks_bersih
0,8m,@alung_koh21471,Koh Alung krisis air Pemerintah berupaya menan...
1,16m,@BatamPos,"Batampos hingga Hari ke -70, Warga Tanjung Sen..."
2,17m,@ciroalves2055,Rizki Fadillnaldo KRISIS AIR BERSIH? JANGAN AS...


4. Restrukturisasi Kolom Target ADUIN

In [8]:
# Cell 4: Restrukturisasi Kolom Target ADUIN

# Daftar 10 Kategori Isu ADUIN
kategori_isu = [
    'Infrastruktur & Jalan', 'Lingkungan & Sampah', 'Air & Sanitasi', 
    'Bencana & Cuaca', 'Transportasi', 'Pelayanan Publik', 
    'Ekonomi & UMKM', 'Keamanan & Sosial', 'Pendidikan', 'Kesehatan'
]

# 1. Setup Kategori Biner (Nilai awal 0)
for kat in kategori_isu:
    df_clean[kat] = 0

# (Opsional tapi disarankan): Kita bisa langsung memberi label 1 
# berdasarkan asal file CSV-nya untuk memperingan kerja labeling nanti
for idx, row in df_clean.iterrows():
    sumber = str(row['kategori_sumber'])
    if sumber in kategori_isu:
        df_clean.at[idx, sumber] = 1

# 2. Setup Kolom Target NLP dengan nilai Placeholder
df_clean['urgensi'] = 'Belum Dilabeli'
df_clean['sentimen'] = 'Belum Dilabeli'
df_clean['lokasi_insiden'] = 'Tidak diketahui'

# 3. Restrukturisasi Urutan Kolom Akhir
kolom_urut = ['waktu', 'username', 'teks_bersih'] + kategori_isu + ['urgensi', 'sentimen', 'lokasi_insiden']
df_aduin = df_clean[kolom_urut].copy()

# Hapus duplikat jika ada tweet yang sama masuk di dua file CSV berbeda
df_aduin = df_aduin.groupby(['username', 'waktu', 'teks_bersih']).max().reset_index()

print("Blueprint arsitektur tabel ADUIN berhasil dibuat!")
display(df_aduin.head())

Blueprint arsitektur tabel ADUIN berhasil dibuat!


,username,waktu,teks_bersih,Infrastruktur & Jalan,Lingkungan & Sampah,Air & Sanitasi,Bencana & Cuaca,Transportasi,Pelayanan Publik,Ekonomi & UMKM,Keamanan & Sosial,Pendidikan,Kesehatan,urgensi,sentimen,lokasi_insiden
0,@0n091,21h,nama tidak boleh kosong ya rugi lah org lombok...,0,0,0,0,0,0,1,0,0,0,Belum Dilabeli,Belum Dilabeli,Tidak diketahui
1,@0rang_Beruntung,Apr 1,"Ya..! : ngapain sih maksa mulu, gak ngaruh jug...",0,0,0,0,0,0,0,0,1,0,Belum Dilabeli,Belum Dilabeli,Tidak diketahui
2,@0tk0il,TRPCL,"aslina ini mah bener, belom lagi kalo ada 2.5K...",1,0,0,0,0,0,0,0,0,0,Belum Dilabeli,Belum Dilabeli,Tidak diketahui
3,@0urdeas,Mar 28,"lova bahkan tempat wisata pantai pun ramai, al...",0,1,0,0,0,0,0,0,0,0,Belum Dilabeli,Belum Dilabeli,Tidak diketahui
4,@1_dreamchaser_1,Mar 28,pupuppp Bersepeda juga bantu kurangi polusi udara,0,1,0,0,0,0,0,0,0,0,Belum Dilabeli,Belum Dilabeli,Tidak diketahui


In [9]:
# Cell 5: Ekspor ke Folder Processed

# Tentukan lokasi folder dan nama file output
output_folder = '../data/processed/'
output_filename = 'twitter_keluhan_processed.csv'
final_path = os.path.join(output_folder, output_filename)

# 1. Pastikan folder 'processed' sudah ada, jika belum maka buat foldernya
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"Folder {output_folder} berhasil dibuat.")

# 2. Ekspor DataFrame ke CSV
# index=False agar kolom indeks bawaan pandas tidak ikut tersimpan sebagai kolom baru
df_aduin.to_csv(final_path, index=False)

print(f"Sukses! Data hasil pembersihan berhasil diekspor ke: {final_path}")
print(f"Jumlah baris yang disimpan: {len(df_aduin)} baris.")

# Tampilkan 5 baris terakhir untuk verifikasi akhir
display(df_aduin.tail())

Sukses! Data hasil pembersihan berhasil diekspor ke: ../data/processed/twitter_keluhan_processed.csv
Jumlah baris yang disimpan: 1771 baris.


,username,waktu,teks_bersih,Infrastruktur & Jalan,Lingkungan & Sampah,Air & Sanitasi,Bencana & Cuaca,Transportasi,Pelayanan Publik,Ekonomi & UMKM,Keamanan & Sosial,Pendidikan,Kesehatan,urgensi,sentimen,lokasi_insiden
1766,@zulfikarnj17,15m,First Theorem of Isomorphism Udh tau di bekasi...,0,0,0,0,1,0,0,0,0,0,Belum Dilabeli,Belum Dilabeli,Tidak diketahui
1767,@zullifkar_ab,MedanEkspress.Com,"Wujudkan Keamanan Berkendara, Babinsa Koramil ...",1,0,0,0,0,0,0,0,0,0,Belum Dilabeli,Belum Dilabeli,Tidak diketahui
1768,@zy_zy_lestary,Apr 1,"ziii air keruh Tiap kali ada celah, hobinya se...",0,0,1,0,0,0,0,0,0,0,Belum Dilabeli,Belum Dilabeli,Tidak diketahui
1769,@zyoel12,Mar 31,zy putus sekolah 1.0 ??? lho tpi waktu live ti...,0,0,0,0,0,0,0,0,1,0,Belum Dilabeli,Belum Dilabeli,Tidak diketahui
1770,@zzhkris,Kris Zzh,Hujan angin plus lampu mati . Lengkap sudah Apr 1,1,0,0,0,0,0,0,0,0,0,Belum Dilabeli,Belum Dilabeli,Tidak diketahui
